# FEEDIT Dictionary DB Load

Django ORM을 직접 사용해서 Dictionary 데이터를 RDS에 적재하는 노트북.

- 각 Dictionary 종류별로 셀 분리
- CSV / XLSX / XLSM 지원
- 현재는 **Category 셀부터 실제 적재 가능**
- 나머지 Dictionary 셀은 같은 노트북에서 순차적으로 추가/실행


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

candidates = [
    PROJECT_ROOT / "backend",
    PROJECT_ROOT,
    PROJECT_ROOT.parent / "backend",
]

BACKEND_DIR = None
for p in candidates:
    if (p / "manage.py").exists():
        BACKEND_DIR = p.resolve()
        break

if BACKEND_DIR is None:
    raise FileNotFoundError("backend/manage.py 위치를 찾지 못했습니다.")

sys.path.insert(0, str(BACKEND_DIR))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

# Jupyter에서 sync ORM 허용
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

print("BACKEND_DIR =", BACKEND_DIR)
print("Django setup 완료")

BACKEND_DIR = C:\SKN31-FINAL-4Team\backend
Django setup 완료


In [2]:
# 1. 공통 유틸: CSV / Excel 읽기
import csv
from pathlib import Path
from openpyxl import load_workbook


def clean(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        return value or None
    return value


def read_table(file_path, sheet_name=None, encoding=None):
    path = Path(file_path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        encodings = [encoding] if encoding else ["utf-8-sig", "utf-8", "cp949"]

        last_error = None
        for enc in encodings:
            try:
                with path.open("r", encoding=enc, newline="") as f:
                    reader = csv.DictReader(f)
                    rows = []
                    for row in reader:
                        item = {
                            str(k).strip(): clean(v)
                            for k, v in row.items()
                            if k is not None
                        }
                        if any(v is not None for v in item.values()):
                            rows.append(item)
                    return rows
            except UnicodeDecodeError as e:
                last_error = e

        raise last_error

    if suffix in {".xlsx", ".xlsm"}:
        wb = load_workbook(path, read_only=True, data_only=True)

        if sheet_name is None:
            sheet_name = wb.sheetnames[0]

        if sheet_name not in wb.sheetnames:
            raise ValueError(
                f"시트 없음: {sheet_name} / 사용 가능: {wb.sheetnames}"
            )

        ws = wb[sheet_name]
        values = ws.iter_rows(values_only=True)
        headers = [clean(v) for v in next(values)]

        rows = []
        for row in values:
            item = {
                headers[i]: clean(row[i])
                for i in range(min(len(headers), len(row)))
                if headers[i]
            }
            if any(v is not None for v in item.values()):
                rows.append(item)

        return rows

    raise ValueError("지원 형식: .csv, .xlsx, .xlsm")


def preview(rows, n=5):
    print("rows =", len(rows))
    for row in rows[:n]:
        print(row)


## CATEGORY

In [8]:
# 2. CATEGORY 적재
from django.db import transaction
from apps.core.models_dictionary import Category

CATEGORY_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_CATEGORY.csv"
)

category_rows = read_table(CATEGORY_FILE)
preview(category_rows)

rows = 78
{'id': None, 'category_code': 'TOP', 'parent_category_code': None, 'name': '상의', 'level': '1', 'sort_order': '10', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'OUTER', 'parent_category_code': None, 'name': '아우터', 'level': '1', 'sort_order': '20', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'BOTTOM', 'parent_category_code': None, 'name': '하의', 'level': '1', 'sort_order': '30', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'DRESS_SKIRT', 'parent_category_code': None, 'name': '원피스·스커트', 'level': '1', 'sort_order': '40', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'category_code': 'SHOES', 'parent_category_code': None, 'name': '신발', 'level': '1', 'sort_order': '50', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}


In [9]:
preview_rows = []

for row in category_rows:
    code = clean(row.get("category_code") or row.get("code"))
    parent_code = clean(row.get("parent_category_code"))

    if not code:
        continue

    item = {
        "code": code,
        "name": clean(row.get("name")),
        "parent_code": parent_code,
        "level": int(row.get("level") or 1),
        "sort_order": (
            int(row["sort_order"])
            if clean(row.get("sort_order")) is not None
            else None
        ),
        "status": clean(row.get("status")) or "ACTIVE",
    }

    preview_rows.append(item)

print(f"적재 예정 건수: {len(preview_rows)}")
print("-" * 80)

for item in preview_rows:
    print(item)

적재 예정 건수: 78
--------------------------------------------------------------------------------
{'code': 'TOP', 'name': '상의', 'parent_code': None, 'level': 1, 'sort_order': 10, 'status': 'ACTIVE'}
{'code': 'OUTER', 'name': '아우터', 'parent_code': None, 'level': 1, 'sort_order': 20, 'status': 'ACTIVE'}
{'code': 'BOTTOM', 'name': '하의', 'parent_code': None, 'level': 1, 'sort_order': 30, 'status': 'ACTIVE'}
{'code': 'DRESS_SKIRT', 'name': '원피스·스커트', 'parent_code': None, 'level': 1, 'sort_order': 40, 'status': 'ACTIVE'}
{'code': 'SHOES', 'name': '신발', 'parent_code': None, 'level': 1, 'sort_order': 50, 'status': 'ACTIVE'}
{'code': 'ACCESSORY', 'name': '잡화·액세서리', 'parent_code': None, 'level': 1, 'sort_order': 60, 'status': 'ACTIVE'}
{'code': 'BAG', 'name': '가방', 'parent_code': None, 'level': 1, 'sort_order': 70, 'status': 'ACTIVE'}
{'code': 'ETC', 'name': '기타', 'parent_code': None, 'level': 1, 'sort_order': 80, 'status': 'ACTIVE'}
{'code': 'TOP_SHORT_SLEEVE_TSHIRT', 'name': '반소매 티셔츠', 'parent_cod

In [10]:
import os
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

In [11]:
created = 0
updated = 0
skipped = 0

with transaction.atomic():

    for row in category_rows:
        code = clean(row.get("category_code") or row.get("code"))

        if not code:
            skipped += 1
            continue

        obj, was_created = Category.objects.update_or_create(
            code=code,
            defaults={
                "name": clean(row.get("name")),
                "level": int(row.get("level") or 1),
                "sort_order": (
                    int(row["sort_order"])
                    if clean(row.get("sort_order")) is not None
                    else None
                ),
                "status": clean(row.get("status")) or Category.Status.ACTIVE,
            },
        )

        created += int(was_created)
        updated += int(not was_created)

    parent_linked = 0

    for row in category_rows:
        code = clean(row.get("category_code") or row.get("code"))
        parent_code = clean(row.get("parent_category_code"))

        if not code or not parent_code:
            continue

        obj = Category.objects.get(code=code)
        parent = Category.objects.get(code=parent_code)

        if obj.pk == parent.pk:
            raise ValueError(f"자기 자신을 parent로 지정할 수 없음: {code}")

        if obj.parent_id != parent.id:
            obj.parent = parent
            obj.save(update_fields=["parent"])
            parent_linked += 1

print({
    "created": created,
    "updated": updated,
    "parent_linked": parent_linked,
    "skipped": skipped,
})

{'created': 78, 'updated': 0, 'parent_linked': 70, 'skipped': 0}


In [12]:
# 4. CATEGORY 적재 확인
qs = (
    Category.objects
    .select_related("parent")
    .order_by("level", "sort_order", "code")
)

print("총 Category:", qs.count())

for c in qs[:30]:
    print(
        c.id,
        c.code,
        c.name,
        "parent=",
        c.parent.code if c.parent else None,
        "level=",
        c.level,
    )


총 Category: 78
1 TOP 상의 parent= None level= 1
2 OUTER 아우터 parent= None level= 1
3 BOTTOM 하의 parent= None level= 1
4 DRESS_SKIRT 원피스·스커트 parent= None level= 1
5 SHOES 신발 parent= None level= 1
6 ACCESSORY 잡화·액세서리 parent= None level= 1
7 BAG 가방 parent= None level= 1
8 ETC 기타 parent= None level= 1
67 ACC_HAT 모자 parent= ACCESSORY level= 2
66 BAG_ALL 가방 parent= BAG level= 2
39 BOTTOM_DENIM 데님 팬츠 parent= BOTTOM level= 2
48 DRESS_MINI 미니 원피스 parent= DRESS_SKIRT level= 2
78 ETC_INNERWEAR 이너웨어 parent= ETC level= 2
19 OUTER_WINDBREAKER 바람막이 parent= OUTER level= 2
57 SHOES_SNEAKERS 스니커즈 parent= SHOES level= 2
9 TOP_SHORT_SLEEVE_TSHIRT 반소매 티셔츠 parent= TOP level= 2
68 ACC_JEWELRY 주얼리 parent= ACCESSORY level= 2
40 BOTTOM_COTTON_CHINO 코튼/치노 팬츠 parent= BOTTOM level= 2
49 DRESS_MIDI 미디 원피스 parent= DRESS_SKIRT level= 2
20 OUTER_TRAINING_JACKET 트레이닝 재킷 parent= OUTER level= 2
58 SHOES_SPORTS 스포츠화 parent= SHOES level= 2
10 TOP_LONG_SLEEVE_TSHIRT 긴소매 티셔츠 parent= TOP level= 2
69 ACC_EYEWEAR 아이웨어 parent= ACCES

## DICTIONARY_TERM

In [3]:
import os
import sys
from pathlib import Path

BACKEND_DIR = Path(r"C:\SKN31-FINAL-4Team\backend")

sys.path.insert(0, str(BACKEND_DIR))

os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

In [6]:

created = 0
updated = 0
skipped = 0

STYLE_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_STYLE.csv"
)
style_rows = read_table(STYLE_FILE)
preview(style_rows)

created = 0
updated = 0
skipped = 0

for row in style_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": clean(row.get("term_type")) or "STYLE",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("normalized_name"))
            or clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "description": clean(row.get("description")),
            "status": clean(row.get("status")) or "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

rows = 14
{'term_id': '1', 'term_code': 'STYLE_BALLETCORE', 'term_type': 'STYLE', 'canonical_name': '발레코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '2', 'term_code': 'STYLE_GORPCORE', 'term_type': 'STYLE', 'canonical_name': '고프코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '3', 'term_code': 'STYLE_BLOKECORE', 'term_type': 'STYLE', 'canonical_name': '블록코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '4', 'term_code': 'STYLE_BIKERCORE', 'term_type': 'STYLE', 'canonical_name': '바이크코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '5', 'term_code': 'STYLE_GEEK_SHEEK', 'term_type': 'STYLE', 'canonical_name': '긱시크', 'style_group': '오피스/무드', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'created': 14, 'updated': 0, 'skipped': 0}


## 스타일 적재


In [7]:
STYLE_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_STYLE.csv"
)
style_rows = read_table(STYLE_FILE)
preview(style_rows)


created = 0
updated = 0
skipped = 0

for row in style_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    raw_is_core = clean(row.get("is_core"))

    is_core = str(raw_is_core).strip().lower() in {
        "1",
        "true",
        "t",
        "y",
        "yes",
        "on",
    }

    obj, was_created = Style.objects.update_or_create(
        term=term,
        defaults={
            "style_group": clean(row.get("style_group")),
            "is_core": is_core,
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

for s in (
    Style.objects
    .select_related("term")
    .order_by("term__term_code")
):
    print(
        s.term_id,
        s.term.term_code,
        s.term.canonical_name,
        s.style_group,
        s.is_core,
    )


rows = 14
{'term_id': '1', 'term_code': 'STYLE_BALLETCORE', 'term_type': 'STYLE', 'canonical_name': '발레코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '2', 'term_code': 'STYLE_GORPCORE', 'term_type': 'STYLE', 'canonical_name': '고프코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '3', 'term_code': 'STYLE_BLOKECORE', 'term_type': 'STYLE', 'canonical_name': '블록코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '4', 'term_code': 'STYLE_BIKERCORE', 'term_type': 'STYLE', 'canonical_name': '바이크코어', 'style_group': '코어(하위문화 믹스)', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'term_id': '5', 'term_code': 'STYLE_GEEK_SHEEK', 'term_type': 'STYLE', 'canonical_name': '긱시크', 'style_group': '오피스/무드', 'is_core': 'TRUE', 'created_at': None, 'updated_at': None}
{'created': 14, 'updated': 0, 'skipped': 0}
11 STYLE_AMEKAJI 아메카

## TERM_ALIAS

In [ ]:
# TERM_ALIAS 적재 셀
# 다음 단계에서 이 셀에 TERM_ALIAS 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# TERM_ALIAS_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# term_alias_rows = read_table(TERM_ALIAS_FILE, sheet_name="...")
# preview(term_alias_rows)

print("TERM_ALIAS - 아직 실행하지 않음")


## STYLE

In [ ]:
# STYLE 적재 셀
# 다음 단계에서 이 셀에 STYLE 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# STYLE_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# style_rows = read_table(STYLE_FILE, sheet_name="...")
# preview(style_rows)

print("STYLE - 아직 실행하지 않음")


## ITEM

In [ ]:
# ITEM 적재 셀
# 다음 단계에서 이 셀에 ITEM 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# ITEM_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# item_rows = read_table(ITEM_FILE, sheet_name="...")
# preview(item_rows)

print("ITEM - 아직 실행하지 않음")


## DETAIL

In [ ]:
# DETAIL 적재 셀
# 다음 단계에서 이 셀에 DETAIL 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# DETAIL_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# detail_rows = read_table(DETAIL_FILE, sheet_name="...")
# preview(detail_rows)

print("DETAIL - 아직 실행하지 않음")


## MATERIAL

In [ ]:
# MATERIAL 적재 셀
# 다음 단계에서 이 셀에 MATERIAL 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# MATERIAL_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# material_rows = read_table(MATERIAL_FILE, sheet_name="...")
# preview(material_rows)

print("MATERIAL - 아직 실행하지 않음")


## COLOR

In [3]:
from apps.core.models_dictionary import DictionaryTerm

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_COLOR.csv"
)
color_rows = read_table(FILE)
preview(color_rows)


created = 0
updated = 0
skipped = 0

for row in color_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    obj, was_created = DictionaryTerm.objects.update_or_create(
        term_code=term_code,
        defaults={
            "term_type": "COLOR",
            "canonical_name": clean(row.get("canonical_name")),
            "normalized_name": clean(row.get("canonical_name")),
            "english_name": clean(row.get("english_name")),
            "status": "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})


rows = 82
{'term_id': None, 'term_code': 'COLOR_WHITE', 'term_type': 'COLOR', 'canonical_name': '화이트', 'english_name': 'White', 'color_family': '뉴트럴', 'base_color_id': None, 'base_color_code': None, 'base_color_name': None, 'note': '가장 기본적인 무채색, 미니멀·클린걸룩의 기본 컬러', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'COLOR_OFF_WHITE', 'term_type': 'COLOR', 'canonical_name': '오프화이트', 'english_name': 'Off-White', 'color_family': '뉴트럴', 'base_color_id': None, 'base_color_code': None, 'base_color_name': None, 'note': '순백보다 은은하게 아이보리 기가 도는 화이트', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'COLOR_IVORY', 'term_type': 'COLOR', 'canonical_name': '아이보리', 'english_name': 'Ivory', 'color_family': '뉴트럴', 'base_color_id': None, 'base_color_code': None, 'base_color_name': None, 'note': '부드럽고 따뜻한 인상의 크림빛 화이트', 'created_at': None, 'updated_at': None}
{'term_id': None, 'term_code': 'COLOR_CREAM', 'term_type': 'COLOR', 'canonical_name': '크림', 'english_name': '

In [4]:
from apps.core.models_dictionary import DictionaryTerm, Color

created = 0
updated = 0
skipped = 0

for row in color_rows:
    term_code = clean(row.get("term_code"))

    if not term_code:
        skipped += 1
        continue

    term = DictionaryTerm.objects.get(
        term_code=term_code
    )

    obj, was_created = Color.objects.update_or_create(
        term=term,
        defaults={
            "color_family": clean(row.get("color_family")),
            "note": clean(row.get("note")),
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

{'created': 82, 'updated': 0, 'skipped': 0}


## TPO

In [ ]:
from apps.core.models_dictionary import DictionaryTerm

FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_TPO.csv"
)
tpo_rows = read_table(FILE)
preview(tpo_rows)



## BRAND

In [3]:
BRAND_CATEGORY_MAP = {
    "국내 캐주얼·스트릿": "BRAND_DOMESTIC_CASUAL_STREET",
    "신발·스니커즈": "BRAND_SHOES_SNEAKERS",
    "국내 컨템포러리·디자이너": "BRAND_DOMESTIC_CONTEMPORARY_DESIGNER",
    "해외 럭셔리·컨템포러리": "BRAND_GLOBAL_LUXURY_CONTEMPORARY",
    "글로벌 스포츠·캐주얼": "BRAND_GLOBAL_SPORTS_CASUAL",
    "스포츠·액티브": "BRAND_SPORTS_ACTIVE",
    "아웃도어": "BRAND_OUTDOOR",
    "골프·스포츠라이프": "BRAND_GOLF_SPORTS_LIFE",
    "키즈": "BRAND_KIDS",
    "라이선스·콜라보": "BRAND_LICENSE_COLLAB",
    "국내 SPA·베이직": "BRAND_DOMESTIC_SPA",
    "무신사 자체 브랜드": "BRAND_MUSINSA",
}

print(len(BRAND_CATEGORY_MAP))

12


In [4]:
from apps.core.models_dictionary import Brand, Category

created = 0
updated = 0
skipped = 0

BRAND_FILE = Path(
    r"C:\SKN31-FINAL-4Team\backend\temp_data\[DB_DICT]_BRAND.csv"
)
brand_rows = read_table(BRAND_FILE)
preview(brand_rows)



for row in brand_rows:

    brand_code = clean(row.get("brand_code"))
    name = clean(row.get("canonical_name"))
    raw_category = clean(row.get("category"))

    if not brand_code or not name:
        skipped += 1
        continue

    category = None

    if raw_category:
        category_code = BRAND_CATEGORY_MAP.get(raw_category)

        if not category_code:
            print("⚠️ 알 수 없는 브랜드 카테고리:", raw_category)
            skipped += 1
            continue

        category = Category.objects.get(
            category_type="BRAND",
            code=category_code,
        )

    obj, was_created = Brand.objects.update_or_create(
        brand_code=brand_code,
        defaults={
            "name": name,
            "english_name": clean(row.get("english_name")),
            "country_code": clean(row.get("country_code")),
            "category": category,
            "description": clean(row.get("description")),
            "status": clean(row.get("status")) or "ACTIVE",
        },
    )

    created += int(was_created)
    updated += int(not was_created)

print({
    "created": created,
    "updated": updated,
    "skipped": skipped,
})

rows = 2775
{'id': None, 'brand_code': 'BRAND_GAKKAI_UNIONS', 'canonical_name': '가까이 유니언즈', 'normalized_name': '가까이 유니언즈', 'english_name': 'GAKKAI UNIONS', 'country_code': None, 'category': '국내 캐주얼·스트릿', 'description': None, 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'brand_code': 'BRAND_GAP', 'canonical_name': '갭', 'normalized_name': '갭', 'english_name': 'GAP', 'country_code': None, 'category': '글로벌 스포츠·캐주얼', 'description': '신발겸업', 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'brand_code': 'BRAND_GUESS', 'canonical_name': '게스', 'normalized_name': '게스', 'english_name': 'GUESS', 'country_code': None, 'category': '글로벌 스포츠·캐주얼', 'description': None, 'status': 'ACTIVE', 'created_at': None, 'updated_at': None}
{'id': None, 'brand_code': 'BRAND_KENZO', 'canonical_name': '겐조', 'normalized_name': '겐조', 'english_name': 'KENZO', 'country_code': None, 'category': '해외 럭셔리·컨템포러리', 'description': None, 'status': 'ACTIVE', 'created_at': None, 

## TERM_RELATION

In [ ]:
# TERM_RELATION 적재 셀
# 다음 단계에서 이 셀에 TERM_RELATION 전용 FK/컬럼 매핑 로직을 넣으면 됨.
# CATEGORY와 독립적으로 실행 가능하도록 셀을 분리해 둠.

# 예:
# TERM_RELATION_FILE = BACKEND_DIR / "data" / "dictionary" / "..."
# term_relation_rows = read_table(TERM_RELATION_FILE, sheet_name="...")
# preview(term_relation_rows)

print("TERM_RELATION - 아직 실행하지 않음")
